# Sharkforce PySpark Smoke Test
Run this first to verify local Spark works.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('sharkforce-pyspark-lab')
    .master('local[*]')
    .config('spark.sql.warehouse.dir', '/workspace/data/spark-warehouse')
    .config('spark.driver.host', '127.0.0.1')
    .config('spark.driver.bindAddress', '127.0.0.1')
    .getOrCreate()
)

print('Spark version:', spark.version)


In [ ]:
records = [
    (1, 'Sean Girgis', 'Murphy', 'TX'),
    (2, 'Sean L Girgis', 'Murphy', 'TX'),
    (3, 'John Smith', 'Dallas', 'TX'),
]
columns = ['record_id', 'full_name', 'city', 'state']
df = spark.createDataFrame(records, columns)
clean_df = (
    df
    .withColumn('name_normalized', F.lower(F.regexp_replace('full_name', '[^a-zA-Z0-9]', '')))
    .withColumn('city_normalized', F.lower(F.trim(F.col('city'))))
)
clean_df.show(truncate=False)


In [ ]:
parquet_path = '/workspace/data/parquet/smoke_people'
clean_df.write.mode('overwrite').parquet(parquet_path)
spark.read.parquet(parquet_path).show(truncate=False)
